In [288]:
#LIBRERY IMPORTS

import pandas as pd

In [289]:
# CREAZIONE DATASET TRANSFERMARKT STAGIONALE

# 1. Caricamento
players = pd.read_csv('players.csv')
players_valutations = pd.read_csv('player_valuations.csv')

# Definiamo i campionati che ci interessano (Big Five)
campionati_target = ['IT1', 'ES1', 'GB1', 'FR1', 'L1']

# Applichiamo il doppio filtro: No Portieri E Solo campionati selezionati
players = players[
    (players['position'] != 'Goalkeeper') & 
    (players['current_club_domestic_competition_id'].isin(campionati_target))
]

# 2. Elaborazione Valutazioni per STAGIONE
players_valutations['date'] = pd.to_datetime(players_valutations['date'])

# Funzione per mappare la data alla stagione di inizio
# Esempio: 09-2023 -> 2023 | 02-2024 -> 2023
def get_season_start(date):
    if date.month >= 7:
        return date.year
    else:
        return date.year - 1

players_valutations['valuation_season'] = players_valutations['date'].apply(get_season_start)

# Calcolo media stagionale e conversione in INTERO (int)
# Usiamo round() prima di astype(int) per non perdere precisione per eccesso/difetto
df_val_avg = players_valutations.groupby(['player_id', 'valuation_season'])['market_value_in_eur'].mean().round().reset_index()

# 3. Elaborazione Anagrafica
players['date_of_birth'] = pd.to_datetime(players['date_of_birth'])
players['born'] = players['date_of_birth'].dt.year.fillna(0).astype(int)

# 4. Merge Selettivo
df_tm_unito = pd.merge(
    df_val_avg,
    players[['player_id', 'name', 'born']], 
    on='player_id', 
    how='inner'
)

# Verifica la pulizia del dataset
print("Dataset Transfermarkt (Stagionale) pronto:")
# print(f"Esempio: Una valutazione del 02-2024 è ora raggruppata sotto l'anno {df_val_avg['valuation_season'].iloc[0]}")
display(df_tm_unito.head())

# Inserisci qui il nome o parte del nome del giocatore
cerca_nome = "lionel messi" 

# Filtro dinamico (non importa se scrivi maiuscolo o minuscolo)
risultato_ricerca = df_tm_unito[df_tm_unito['name'].str.contains(cerca_nome, case=False, na=False)]

if not risultato_ricerca.empty:
    print(f"Risultati trovati per '{cerca_nome}':")
    # Mostriamo solo alcune colonne chiave per leggibilità
    display(risultato_ricerca[['player_id', 'valuation_season', 'market_value_in_eur', 'name', 'born']].sort_values('valuation_season'))
else:
    print(f"Nessun giocatore trovato con il nome: {cerca_nome}")

Dataset Transfermarkt (Stagionale) pronto:


,player_id,valuation_season,market_value_in_eur,name,born
0,10,2004,9333333.0,Miroslav Klose,1978
1,10,2005,17500000.0,Miroslav Klose,1978
2,10,2006,26500000.0,Miroslav Klose,1978
3,10,2007,20000000.0,Miroslav Klose,1978
4,10,2008,18000000.0,Miroslav Klose,1978


Risultati trovati per 'lionel messi':


,player_id,valuation_season,market_value_in_eur,name,born
10323,28003,2004,3000000.0,Lionel Messi,1987
10324,28003,2005,10000000.0,Lionel Messi,1987
10325,28003,2007,51666667.0,Lionel Messi,1987
10326,28003,2008,56666667.0,Lionel Messi,1987
10327,28003,2009,75000000.0,Lionel Messi,1987
10328,28003,2010,100000000.0,Lionel Messi,1987
10329,28003,2011,100000000.0,Lionel Messi,1987
10330,28003,2012,120000000.0,Lionel Messi,1987
10331,28003,2013,120000000.0,Lionel Messi,1987
10332,28003,2014,120000000.0,Lionel Messi,1987


In [290]:
# 1. Carichiamo il dataset delle statistiche
df_stats_all = pd.read_csv("All_Players_1992-2025.csv")

# 2. Pulizia della colonna Season
# Se Season è "2022-2023", prendiamo i primi 4 caratteri e convertiamo in numero
df_stats_all['match_year'] = df_stats_all['Season'].str[:4].astype(int)

# 3. Assicuriamoci che 'born' sia dello stesso tipo (int) in entrambi i dataframe
# Nel tuo screenshot era un float (1978.0), lo rendiamo intero per sicurezza
df_tm_unito['born'] = df_tm_unito['born'].fillna(0).astype(int)
df_stats_all['Born'] = df_stats_all['Born'].fillna(0).astype(int)

# 4. IL MERGE FINALE
# Uniamo per Nome, Anno di Nascita e Anno della Stagione
df_final_dataset = pd.merge(
    df_stats_all, 
    df_tm_unito, 
    left_on=['Player', 'Born', 'match_year',], 
    right_on=['name', 'born', 'valuation_season'], 
    how='inner'
)

# 5. Pulizia post-merge: rimuoviamo le colonne doppie
df_final_dataset = df_final_dataset.drop(columns=['name', 'born','player_id'])

print(f"Dataset finale creato! Totale righe: {len(df_final_dataset)}")
display(df_final_dataset.head(10))

Dataset finale creato! Totale righe: 32256


,PlayerID,Player,Squad,League,Nation,Pos,Age,Born,Season,MP,...,UCL_Gls,UCL_xG,UCL_Ast,UCL_xA,UCL_KP,UCL_GCA,UCL_SCA,match_year,valuation_season,market_value_in_eur
0,1585,Levan Kobiashvili,Hertha BSC,Bundesliga,GEO,"DF,MF",32.0,1977,2009-2010,16.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2009,2009,1666667.0
1,1585,Levan Kobiashvili,Hertha BSC,Bundesliga,GEO,"DF,MF",34.0,1977,2011-2012,31.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2011,2011,625000.0
2,1585,Levan Kobiashvili,Schalke 04,Bundesliga,GEO,"DF,MF",28.0,1977,2005-2006,32.0,...,3.0,0.0,1.0,0.0,0.0,0.0,0.0,2005,2005,6666667.0
3,1585,Levan Kobiashvili,Schalke 04,Bundesliga,GEO,"DF,MF",30.0,1977,2007-2008,13.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2007,2007,4500000.0
4,1585,Levan Kobiashvili,Schalke 04,Bundesliga,GEO,"DF,MF",27.0,1977,2004-2005,32.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2004,2004,3250000.0
5,1585,Levan Kobiashvili,Schalke 04,Bundesliga,GEO,"DF,MF",31.0,1977,2008-2009,29.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2008,2008,3000000.0
6,1585,Levan Kobiashvili,Schalke 04,Bundesliga,GEO,"DF,MF",32.0,1977,2009-2010,4.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2009,2009,1666667.0
7,1599,Martin Stranzl,Gladbach,Bundesliga,AUT,"DF,MF",30.0,1980,2010-2011,17.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2010,2010,3000000.0
8,1599,Martin Stranzl,Gladbach,Bundesliga,AUT,"DF,MF",31.0,1980,2011-2012,22.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2011,2011,3000000.0
9,1599,Martin Stranzl,Gladbach,Bundesliga,AUT,"DF,MF",32.0,1980,2012-2013,26.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2012,2012,2500000.0


In [291]:
print(df_final_dataset.columns.tolist())

['PlayerID', 'Player', 'Squad', 'League', 'Nation', 'Pos', 'Age', 'Born', 'Season', 'MP', 'Min', 'Mn/MP', 'Min%', 'Starts', 'Mn/Start', 'Subs', 'Mn/Sub', 'unSub', '90s', 'Sh', 'Sh/90', 'SoT', 'SoT%', 'SoT/90', 'G/Sh', 'G/SoT', 'Gls', 'Ast', 'G+A', 'PK', 'PKatt', 'PKcon', 'OG', 'xG', 'npxG', 'npxG/Sh', 'G-xG', 'np:G-xG', 'Pass', 'Cmp', 'Cmp%', 'PassLive', 'PassDead', 'KP', 'Crs', 'CrsPA', 'A-xAG', 'xAG', 'xA', 'PPA', 'Live', 'Dead', 'FK', 'TB', 'Sw', 'TI', 'CK', 'In', 'Out', 'Str', 'Compl', 'PPM', 'Mis', 'Tkl', 'TklW', 'Tkl%', 'Tkld', 'Tkld%', 'Tkl+Int', 'Int', 'Blocks', 'Clr', 'Fls', 'Recov', 'Def', 'Def 3rd', 'Mid 3rd', 'Att 3rd', 'Att', 'Att Pen', 'Off', 'Dis', 'Won', 'Won%', 'Lost', '+/-', '+/-90', 'On-Off', 'onG', 'onGA', 'onxG', 'onxGA', 'xG+/-', 'xG+/-90', 'SCA', 'SCA90', 'PrgC', 'PrgDist', 'PrgP', 'PrgR', 'Rec', 'Carries', 'CPA', 'Touches', 'Dist', 'TotDist', 'Ballon d or', 'European Golden Shoe', 'League Won', 'UCL_Won', 'The Best FIFA Mens Player', 'UEFA Best Player', 'UCL_MP'

In [292]:
# Informazioni generali sul dataset
print(f"--- INFO DATASET FINALE ---")
print(f"Numero totale di righe (osservazioni stagionali): {len(df_final_dataset)}")
print(f"Numero di colonne disponibili: {df_final_dataset.shape[1]}")

# Conteggio dei giocatori univoci
n_giocatori = df_final_dataset['PlayerID'].nunique()
print(f"Numero totale di giocatori unici mappati: {n_giocatori}")

# Distribuzione delle stagioni per giocatore (quante volte appare ogni giocatore)
print("\nTop 10 giocatori con più stagioni nel dataset:")
stagioni_per_giocatore = df_final_dataset.groupby(['PlayerID', 'Player']).size().sort_values(ascending=False)
display(stagioni_per_giocatore.head(10))

# Calcolo della media delle stagioni per giocatore
print(f"\nMedia di stagioni registrate per giocatore: {stagioni_per_giocatore.mean():.2f}")

--- INFO DATASET FINALE ---
Numero totale di righe (osservazioni stagionali): 32256
Numero di colonne disponibili: 123
Numero totale di giocatori unici mappati: 7008

Top 10 giocatori con più stagioni nel dataset:


PlayerID  Player           
17370     James Milner         23
7943      Raúl Albiol          21
8498      Cristiano Ronaldo    20
18388     Marco Borriello      20
17709     Ashley Young         20
7944      Raúl García          20
12623     Mathieu Debuchy      20
18355     Fernandinho          20
13111     Moussa Sissoko       19
8116      César Azpilicueta    19
dtype: int64


Media di stagioni registrate per giocatore: 4.60


In [293]:
# Inserisci qui il nome o parte del nome del giocatore
cerca_nome = "" 

# Filtro dinamico (non importa se scrivi maiuscolo o minuscolo)
risultato_ricerca = df_final_dataset[df_final_dataset['Player'].str.contains(cerca_nome, case=False, na=False)]

if not risultato_ricerca.empty:
    print(f"Risultati trovati per '{cerca_nome}':")
    # Mostriamo solo alcune colonne chiave per leggibilità
    display(risultato_ricerca[['PlayerID', 'Player', 'Nation', 'Season', 'Born', 'market_value_in_eur', 'Gls', 'Ast', 'Squad','match_year', 'valuation_season', 'Pos']].sort_values('Season'))
else:
    print(f"Nessun giocatore trovato con il nome: {cerca_nome}")

Risultati trovati per '':


,PlayerID,Player,Nation,Season,Born,market_value_in_eur,Gls,Ast,Squad,match_year,valuation_season,Pos
16078,12331,Benoît Pedretti,FRA,2004-2005,1980,7500000.0,3.0,2.0,Marseille,2004,2004,MF
27598,21865,Rodrigo Taddei,BRA,2004-2005,1980,5500000.0,3.0,2.0,Siena,2004,2004,"DF,FW"
16069,12322,Alaeddine Yahia,TUN,2004-2005,1981,1325000.0,0.0,0.0,Saint-Étienne,2004,2004,DF
16080,12334,Chaouki Ben Saada,TUN,2004-2005,1984,916667.0,1.0,0.0,Bastia,2004,2004,"FW,MF"
16086,12343,Didier Drogba,CIV,2004-2005,1978,30000000.0,10.0,5.0,Chelsea,2004,2004,FW
...,...,...,...,...,...,...,...,...,...,...,...,...
15376,10822,Alberto Moleiro,ESP,2024-2025,2003,22500000.0,6.0,1.0,Las Palmas,2024,2024,"FW,MF"
15377,10823,Alberto Risco,ESP,2024-2025,2005,200000.0,0.0,0.0,Getafe,2024,2024,"MF,FW"
30366,23159,Alessandro Deiola,ITA,2024-2025,1995,1600000.0,2.0,1.0,Cagliari,2024,2024,MF
30351,23158,Alessandro Buongiorno,ITA,2024-2025,1999,45000000.0,1.0,0.0,Napoli,2024,2024,DF


In [294]:
df_final_dataset.to_csv("dataset.csv", index=True)

In [295]:
df_final_dataset["Pos"].unique()

array(['DF,MF', 'FW', 'FW,MF', 'MF', 'DF', 'MF,FW', 'MF,DF', 'DF,FW',
       'FW,DF'], dtype=object)

In [296]:

# Supponiamo che il tuo dataset sia stato caricato nella variabile 'df'
# df = pd.read_csv('tuo_dataset.csv')

# 1. Estrazione del Ruolo Primario e Secondario
# Utilizziamo .str.split() per dividere la stringa in base alla virgola.
# Il parametro expand=True trasforma il risultato in colonne separate.
pos_split = df_final_dataset["Pos"].str.split(',', expand=True)

# Assegniamo la prima colonna generata al Ruolo Primario
df_final_dataset['Primary_Pos'] = pos_split[0].str.strip() 

# Assegniamo la seconda colonna al Ruolo Secondario (se esiste)
# Alcuni giocatori hanno un solo ruolo, quindi gestiamo l'errore se la colonna 1 non esiste
if 1 in pos_split.columns:
    df_final_dataset['Secondary_Pos'] = pos_split[1].str.strip()
    # Riempiamo i valori vuoti (NaN) con la stringa 'None' o 'Nessuno'
    df_final_dataset['Secondary_Pos'] = df_final_dataset['Secondary_Pos'].fillna('NaN')
else:
    df_final_dataset['Secondary_Pos'] = 'NaN'

# 2. Eliminazione della colonna originale (non più utile per il modello)
df_final_dataset = df_final_dataset.drop(columns=['Pos'])

# Visualizziamo il risultato
display(df_final_dataset.head(20))


# ONE-HOT ENCODING SUI RUOLI

,PlayerID,Player,Squad,League,Nation,Age,Born,Season,MP,Min,...,UCL_Ast,UCL_xA,UCL_KP,UCL_GCA,UCL_SCA,match_year,valuation_season,market_value_in_eur,Primary_Pos,Secondary_Pos
0,1585,Levan Kobiashvili,Hertha BSC,Bundesliga,GEO,32.0,1977,2009-2010,16.0,1440.0,...,0.0,0.0,0.0,0.0,0.0,2009,2009,1666667.0,DF,MF
1,1585,Levan Kobiashvili,Hertha BSC,Bundesliga,GEO,34.0,1977,2011-2012,31.0,2751.0,...,0.0,0.0,0.0,0.0,0.0,2011,2011,625000.0,DF,MF
2,1585,Levan Kobiashvili,Schalke 04,Bundesliga,GEO,28.0,1977,2005-2006,32.0,2880.0,...,1.0,0.0,0.0,0.0,0.0,2005,2005,6666667.0,DF,MF
3,1585,Levan Kobiashvili,Schalke 04,Bundesliga,GEO,30.0,1977,2007-2008,13.0,768.0,...,0.0,0.0,0.0,0.0,0.0,2007,2007,4500000.0,DF,MF
4,1585,Levan Kobiashvili,Schalke 04,Bundesliga,GEO,27.0,1977,2004-2005,32.0,2857.0,...,0.0,0.0,0.0,0.0,0.0,2004,2004,3250000.0,DF,MF
5,1585,Levan Kobiashvili,Schalke 04,Bundesliga,GEO,31.0,1977,2008-2009,29.0,2136.0,...,0.0,0.0,0.0,0.0,0.0,2008,2008,3000000.0,DF,MF
6,1585,Levan Kobiashvili,Schalke 04,Bundesliga,GEO,32.0,1977,2009-2010,4.0,244.0,...,0.0,0.0,0.0,0.0,0.0,2009,2009,1666667.0,DF,MF
7,1599,Martin Stranzl,Gladbach,Bundesliga,AUT,30.0,1980,2010-2011,17.0,1530.0,...,0.0,0.0,0.0,0.0,0.0,2010,2010,3000000.0,DF,MF
8,1599,Martin Stranzl,Gladbach,Bundesliga,AUT,31.0,1980,2011-2012,22.0,1781.0,...,0.0,0.0,0.0,0.0,0.0,2011,2011,3000000.0,DF,MF
9,1599,Martin Stranzl,Gladbach,Bundesliga,AUT,32.0,1980,2012-2013,26.0,2273.0,...,0.0,0.0,0.0,0.0,0.0,2012,2012,2500000.0,DF,MF


In [297]:
df_final_dataset['market_value_in_eur_squad_avg'] = df_final_dataset.groupby(['Squad', 'valuation_season'])['market_value_in_eur'].transform('mean').round()

df_final_dataset

,PlayerID,Player,Squad,League,Nation,Age,Born,Season,MP,Min,...,UCL_xA,UCL_KP,UCL_GCA,UCL_SCA,match_year,valuation_season,market_value_in_eur,Primary_Pos,Secondary_Pos,market_value_in_eur_squad_avg
0,1585,Levan Kobiashvili,Hertha BSC,Bundesliga,GEO,32.0,1977,2009-2010,16.0,1440.0,...,0.0,0.0,0.0,0.0,2009,2009,1666667.0,DF,MF,1707292.0
1,1585,Levan Kobiashvili,Hertha BSC,Bundesliga,GEO,34.0,1977,2011-2012,31.0,2751.0,...,0.0,0.0,0.0,0.0,2011,2011,625000.0,DF,MF,2308929.0
2,1585,Levan Kobiashvili,Schalke 04,Bundesliga,GEO,28.0,1977,2005-2006,32.0,2880.0,...,0.0,0.0,0.0,0.0,2005,2005,6666667.0,DF,MF,3079167.0
3,1585,Levan Kobiashvili,Schalke 04,Bundesliga,GEO,30.0,1977,2007-2008,13.0,768.0,...,0.0,0.0,0.0,0.0,2007,2007,4500000.0,DF,MF,4916667.0
4,1585,Levan Kobiashvili,Schalke 04,Bundesliga,GEO,27.0,1977,2004-2005,32.0,2857.0,...,0.0,0.0,0.0,0.0,2004,2004,3250000.0,DF,MF,2105000.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
32251,24710,Tobias Slotsager,Hellas Verona,Serie A,DEN,18.0,2006,2024-2025,0.0,0.0,...,0.0,0.0,0.0,0.0,2024,2024,2000000.0,DF,NaN,3074107.0
32252,24713,Tomás Palacios,Inter,Serie A,ARG,21.0,2003,2024-2025,2.0,11.0,...,0.0,0.0,0.0,0.0,2024,2024,5666667.0,DF,NaN,30933333.0
32253,24713,Tomás Palacios,Monza,Serie A,ARG,21.0,2003,2024-2025,8.0,446.0,...,0.0,0.0,0.0,0.0,2024,2024,5666667.0,DF,NaN,3299107.0
32254,24715,Vasilije Adžić,Juventus,Serie A,MNE,18.0,2006,2024-2025,6.0,42.0,...,0.0,0.0,0.0,0.0,2024,2024,2333333.0,MF,FW,25341667.0


In [298]:
df_final_dataset[["PlayerID", "Player", "Squad", "valuation_season", "Age", "Primary_Pos", "Secondary_Pos", "npxG", "xAG", "PrgC", "PrgP", "SCA", "Tkl+Int", "Recov", "market_value_in_eur", "market_value_in_eur_squad_avg"]].to_csv("godo.csv", index=True)